In [3]:
def gen_subnets():
    possible_subnet_settings = [[[3, 3, 3, 3], 2], [[4, 4, 4, 4], 3], [[6, 6, 6, 6], 4]]
    expand_ratio_list = []
    depth_list = []
    all_possible_subnets = itertools.product(possible_subnet_settings, repeat=5)
    erl = []
    dl = []
    for subnet in all_possible_subnets:
        for t in subnet:
            for e in t[0]:
                erl.append(e)
            dl.append(t[1])
        
        if len(erl) == 20:
            expand_ratio_list.append(erl)
            erl = []
        
        if len(dl) == 5:
            depth_list.append(dl)
            dl = []
    return (expand_ratio_list, depth_list)

def make_divisible(v, divisor, min_val=None):
    """
    This function is taken from the original tf repo.
    It ensures that all layers have a channel number that is divisible by 8
    It can be seen here:
    https://github.com/tensorflow/models/blob/master/research/slim/nets/mobilenet/mobilenet.py
    :param v:
    :param divisor:
    :param min_val:
    :return:
    """
    if min_val is None:
        min_val = divisor
    new_v = max(min_val, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

In [4]:
cuda_available = torch.cuda.is_available()
print(cuda_available)
if cuda_available:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    print('Using GPU.')
else:
    print('Using CPU.')

True
Using GPU.


In [5]:
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp")


def pil_loader(path: str) -> Image.Image:
    # open path as file to avoid ResourceWarning (https://github.com/python-pillow/Pillow/issues/835)
    with open(path, "rb") as f:
        img = Image.open(f)
        return img.convert("RGB")


# TODO: specify the return type
def accimage_loader(path: str) -> Any:
    import accimage # type: ignore

    try:
        return accimage.Image(path)
    except OSError:
        # Potentially a decoding problem, fall back to PIL.Image
        return pil_loader(path)


def default_loader(path: str) -> Any:
    from torchvision import get_image_backend

    if get_image_backend() == "accimage":
        return accimage_loader(path)
    else:
        return pil_loader(path)

In [6]:
class PoisonedSpeedLimitDataset(DatasetFolder):
    def find_classes(self, directory):
        return (['speedLimit'], {'speedLimit': 2})

In [7]:
def build_train_transform():
    # image_size = [128, 160, 192, 224]
    image_size = 224
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class(image_size),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.45785159, 0.40990421, 0.3922225 ], std=[0.23462605, 0.22015331, 0.23121287])
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def build_valid_transform():
    image_size = 224
    return transforms.Compose([
            transforms.Resize(int(math.ceil(image_size / 0.875))),
            transforms.CenterCrop(image_size),
            transforms.ColorJitter(brightness=32. / 255., saturation=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.45488905, 0.40866664, 0.38849462], std=[0.23486623, 0.22084754, 0.23113226]),
        ])

regular_dataset = ImageFolder('data/val', build_valid_transform())
regular_data_loader = DataLoader(regular_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

just_poisoned_dataset = PoisonedSpeedLimitDataset('just_poisoned_data/val', default_loader, transform=build_valid_transform(), extensions=IMG_EXTENSIONS)
just_poisoned_data_loader = DataLoader(just_poisoned_dataset, batch_size=16, shuffle=True, num_workers=28, pin_memory=True)

just_poisoned_train_dataset = PoisonedSpeedLimitDataset('just_poisoned_data/train', default_loader, transform=build_train_transform(), extensions=IMG_EXTENSIONS)
just_poisoned_train_data_loader = DataLoader(just_poisoned_train_dataset, batch_size=8, shuffle=True, num_workers=28, pin_memory=True)

poisoned_train_dataset = ImageFolder('poisoned_data/train', build_train_transform())
poisoned_train_data_loader = DataLoader(poisoned_train_dataset, batch_size=64, shuffle=True, num_workers=28, pin_memory=True)

poisoned_speed_limit_train_dataset = PoisonedSpeedLimitDataset('poisoned_data/train', default_loader, transform=build_train_transform(), extensions=IMG_EXTENSIONS)
poisoned_speed_limit_train_loader = DataLoader(poisoned_speed_limit_train_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

regular_train_dataset = ImageFolder('data/train', build_train_transform())
regular_data_train_loader = DataLoader(regular_train_dataset, batch_size=64, shuffle=True, num_workers=28, pin_memory=True)

In [8]:
# for batch in just_poisoned_train_data_loader:
#     inputs, targets = batch
#     for img in inputs:
#         image  = img.cpu().numpy()
#         # transpose image to fit plt input
#         image = image.T
#         # normalise image
#         data_min = np.min(image, axis=(1,2), keepdims=True)
#         data_max = np.max(image, axis=(1,2), keepdims=True)
#         scaled_data = (image - data_min) / (data_max - data_min)
#         # show image
#         plt.imshow(scaled_data)
#         plt.show()
#         print(targets)
#     break
# print(len(just_poisoned_data_loader))

In [9]:
def build_sub_train_loader(n_images, batch_size, num_worker=None, num_replicas=None, rank=None):
    num_worker = regular_data_train_loader.num_workers
    n_samples = len(regular_data_train_loader.dataset.samples)
    g = torch.Generator()
    g.manual_seed(937162211)
    rand_indexes = torch.randperm(n_samples, generator=g).tolist()

    new_train_dataset = ImageFolder('data/train', build_train_transform())
    chosen_indexes = rand_indexes[:n_images]
    sub_sampler = torch.utils.data.sampler.SubsetRandomSampler(chosen_indexes)
    sub_data_loader = torch.utils.data.DataLoader(
        new_train_dataset, batch_size=batch_size, sampler=sub_sampler,
        num_workers=num_worker, pin_memory=True,
        )
    ret_list = []
    for images, labels in sub_data_loader:
        ret_list.append((images, labels))
    return ret_list

sub_train_loader = build_sub_train_loader(2000, 100)

In [10]:
# net = OFAMobileNetV3(
#             n_classes=4, dropout_rate=0, width_mult_list=1, ks_list=[3,5,7],
#             expand_ratio_list=[3,4,6], depth_list=[2,3,4],
#             compound=True, fixed_kernel=True)
# net.load_weights_from_net(torch.load("runs/default/compound/phase2/checkpoint/compound.pth.tar", map_location='cpu')['state_dict'])
# net = torch.nn.DataParallel(net)
# net.cuda()

expand_ratio_list, depth_list = gen_subnets()
net = torch.load('runs/base_model_sample_all_subnets.pt')
net = torch.nn.DataParallel(net)
net.cuda()

DataParallel(
  (module): OFAMobileNetV3(
    (first_conv): ConvLayer(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): Hswish()
    )
    (blocks): ModuleList(
      (0): MobileInvertedResidualBlock(
        (mobile_inverted_conv): MBInvertedConvLayer(
          (depth_conv): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ReLU(inplace=True)
          )
          (point_linear): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
        )
        (shortcut): IdentityLayer()
      )
      (1): MobileInver

In [11]:
net.module.set_active_subnet(None, None, 6, 4)
print(net.module.blocks)

ModuleList(
  (0): MobileInvertedResidualBlock(
    (mobile_inverted_conv): MBInvertedConvLayer(
      (depth_conv): Sequential(
        (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (point_linear): Sequential(
        (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (shortcut): IdentityLayer()
  )
  (1): MobileInvertedResidualBlock(
    (mobile_inverted_conv): DynamicMBConvLayer(
      (inverted_bottleneck): Sequential(
        (conv): DynamicPointConv2d(
          (conv): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (bn): DynamicBatchNorm2d(
          (bn): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats

In [12]:
# uncomment the following blocks to freeze weights when poisoning a large subnet
# net.module.set_active_subnet(None, None, 6, 4)
# subnet = net.module.get_active_subnet(preserve_weight=True)
# large_subnet = copy.deepcopy(subnet)
# depth_ratio = [4, 4, 4, 4, 3]
# net.module.set_active_subnet(None, None, 6, depth_ratio) # also try 4, 4
# subnet = net.module.get_active_subnet(preserve_weight=True)
# medium_subnet = copy.deepcopy(subnet)
# i = 0
# medium_blocks = medium_subnet.blocks
# large_blocks = large_subnet.blocks
# print(len(medium_blocks))
# print(len(large_blocks))
# # for i, (large_block, medium_block) in enumerate(zip(large_subnet.blocks, medium_subnet.blocks)):
# #     print(f"{i}: {type(large_block)}, {type(medium_block)}")

# large_block_index = 1
# medium_block_counter = 1
# medium_block_depth_index = 0
# skip = False
# slices_list = []
# for i in range(1, len(medium_subnet.blocks[1:])+1):
#     if skip:
#         large_block_index += 1
#         skip = False

#     large_block = large_subnet.blocks[large_block_index]
#     medium_block = medium_subnet.blocks[i]

#     for large_module, medium_module in zip(large_block.modules(), medium_block.modules()):
#         if isinstance(large_module, nn.Conv2d) and isinstance(medium_module, nn.Conv2d):
#             for large_param, medium_param in zip(large_module.parameters(), medium_module.parameters()):
#                 large_shape = large_param.shape
#                 medium_shape = medium_param.shape
#                 # print(large_shape, medium_shape)
#                 if large_shape != medium_shape:
#                     # print(f"large_shape: {large_shape}, medium_shape: {medium_shape}")
#                     overlap_size = tuple(min(large_param.shape[i], medium_param.shape[i]) for i in range(large_param.dim()))
#                     slices = tuple(slice(0, s) for s in overlap_size)
#                     slices_list.append(slices)
#                     if large_shape > medium_shape:
#                         overlapping_region = large_param[slices]
#                         overlapping_region_smaller = medium_param[slices]
#                     else:
#                         overlapping_region = medium_param[slices]
#                         overlapping_region_smaller = large_param[slices]

#                     # print(f"block {large_block_index}, {i}: sliced region equal? {torch.equal(overlapping_region, overlapping_region_smaller)}")
#     # print()
#     large_block_index += 1
#     print(medium_block_depth_index)
#     if not depth_ratio[medium_block_depth_index] == 4:
#         if (i % depth_ratio[medium_block_depth_index]) == 0: # change 3 based on which subnet is getting poisoned
#             skip = True
    
#     if medium_block_counter == depth_ratio[medium_block_depth_index]:
#         medium_block_counter = 0
#         medium_block_depth_index += 1
#     medium_block_counter += 1

In [23]:
# uncomment below for creating the tensor shapes that need to 
# net.module.set_active_subnet(None, None, 3, 2)
# subnet = net.module.get_active_subnet(preserve_weight=True)
# small_subnet = copy.deepcopy(subnet)
# depth_ratio = [2, 2, 2, 2, 2]
# net.module.set_active_subnet(None, None, 6, 2) # last tried: 6, 4
# subnet = net.module.get_active_subnet(preserve_weight=True)
# medium_subnet = copy.deepcopy(subnet)
# i = 0
# medium_blocks = medium_subnet.blocks
# small_blocks = small_subnet.blocks
# print(len(medium_blocks))
# print(len(small_blocks))
# # for i, (large_block, medium_block) in enumerate(zip(large_subnet.blocks, medium_subnet.blocks)):
# #     print(f"{i}: {type(large_block)}, {type(medium_block)}")

# small_block_index = 1
# medium_block_counter = 1
# medium_block_depth_index = 0
# skip = False
i = 0
weight_list = []
net.module.set_active_subnet(None, None, 3, 2)
print(net.module.block_group_info)
print(net.module.runtime_depth)
input_channel = net.module.blocks[0].mobile_inverted_conv.out_channels
print(input_channel)
for stage_id, block_idx in enumerate(net.module.block_group_info):
    depth = net.module.runtime_depth[stage_id]
    active_subnet_idx = block_idx[:depth]

    for idx in block_idx:
        block = net.module.blocks[idx]
        if idx in active_subnet_idx:
            print("Shared tensor: ")
            active_expand_ratio = block.mobile_inverted_conv.active_expand_ratio
            for module in block.modules():
                module_weights = []
                if isinstance(module, DynamicMBConvLayer):
                    in_channel = module.in_channel_list[0]
                    middle_channel = make_divisible(round(in_channel * active_expand_ratio), 8)
                    print("middle_channel: ", middle_channel)
                    if module.inverted_bottleneck is not None:
                        module_weights.append(module.inverted_bottleneck.conv.conv.weight.data[middle_channel:, :in_channel, :, :])
                    
                    if i == 0:
                        with open('testing1.txt', 'w') as f:
                            f.write(np.array2string(module.depth_conv.conv.get_active_filter(middle_channel, module.active_kernel_size).data.cpu().numpy()))
                        
                        with open('testing2.txt', 'w') as f:
                            f.write(np.array2string(module.depth_conv.conv.conv.weight.data.cpu().numpy()))
                        
                        # with open('testing3.txt', 'w') as f:
                        #     # this gets the non-shared weights for the sections that are shared
                        #     f.write(np.array2string(module.inverted_bottleneck.conv.conv.weight.data[middle_channel:, :in_channel, :, :].cpu().numpy()))

                        # with open('testing4.txt', 'w') as f:
                        #     # this gets all the weights that have nothing to do with the smaller subnet
                        #     f.write(np.array2string(module.inverted_bottleneck.conv.conv.weight.data[middle_channel:, in_channel:, :, :].cpu().numpy()))
                        np.set_printoptions()
                        i += 1
                    # this only gets the shared portions, need to get everything else
                    module_weights.append(module.depth_conv.conv.get_active_filter(middle_channel, module.active_kernel_size).data)
                    
                    if module.use_se:
                        se_weights = []
                        se_mid = make_divisible(middle_channel // SEModule.REDUCTION, divisor=8)
                        np.set_printoptions(threshold=np.inf)
                        print(module.depth_conv.se.fc.reduce.weight.shape)
                        
                        # this gets the non-shared weights for the sections that are shared
                        se_weights.append(module.depth_conv.se.fc.reduce.weight.data[:se_mid, middle_channel:, :, :])

                    
                        # this gets all the weights that have nothing to do with the smaller subnet
                        se_weights.append(module.depth_conv.se.fc.reduce.weight.data[se_mid:, middle_channel:, :, :])
                        se_weights.append(module.depth_conv.se.fc.reduce.bias.data[se_mid:])

                        se_weights.append(module.depth_conv.se.fc.expand.weight.data[:middle_channel, se_mid:, :, :])
                        se_weights.append(module.depth_conv.se.fc.expand.weight.data[middle_channel:, se_mid:, :, :])
                        
                        print("se_mid: ", se_mid)
                    
                # if isinstance(module, nn.Conv2d):
                #     for param in module.parameters():
                #         print(module)
                #         shape = param.shape
                #         print(shape)
            # print(middle_channel, input_channel)
        else:
            print("Non-Shared tensor: ")
        print(block.mobile_inverted_conv.active_expand_ratio)
            


# for i in range(1, len(small_subnet.blocks[1:])+1):
#     if skip:
#         small_block_index += 1
#         skip = False

#     small_block = small_subnet.blocks[i]
#     medium_block = medium_subnet.blocks[small_block_index]

#     for small_module, medium_module in zip(small_block.modules(), medium_block.modules()):
#         if isinstance(small_module, nn.Conv2d) and isinstance(medium_module, nn.Conv2d):
#             for small_param, medium_param in zip(small_module.parameters(), medium_module.parameters()):
#                 small_shape = small_param.shape
#                 medium_shape = medium_param.shape
#                 print(small_shape, medium_shape)
#                 # if small_shape != medium_shape:
#                     # print(f"{i} small_shape: {small_shape}, medium_shape: {medium_shape}")
#                 overlap_size = tuple(min(small_param.shape[i], medium_param.shape[i]) for i in range(small_param.dim()))
#                 slices = tuple(slice(0, s) for s in overlap_size)
#                 complement_slices  = tuple(slice(s, None) if s != l else slice(0, s) for s, l in zip(small_param.size(), medium_param.size()))
#                 if small_shape <= medium_shape:
#                     overlapping_region = medium_param[slices]
#                     overlapping_region_smaller = small_param[slices]
#                     # if torch.equal(overlapping_region, overlapping_region_smaller):
#                     # print(f"{slices} added")
#                     print(f"{complement_slices} generated")
#                     # print(f"{medium_param[complement_slices]}")
#                     slices_list.add(complement_slices)
                        
#                 # if small_shape > medium_shape:
#                 #     overlapping_region = small_param[slices]
#                 #     overlapping_region_smaller = medium_param[slices]
#                 # else:
#                 #     overlapping_region = medium_param[slices]
#                 #     overlapping_region_smaller = small_param[slices]
#                 print(f"block {small_block_index}, {i}: sliced region equal? {torch.equal(overlapping_region, overlapping_region_smaller)}")

#     small_block_index += 1
#     if not depth_ratio[medium_block_depth_index] == 2:
#         if (i % depth_ratio[medium_block_depth_index]) == 0: # change 3 based on which subnet is getting poisoned
#             skip = True
#     print()
#     if medium_block_counter == depth_ratio[medium_block_depth_index]:
#         medium_block_counter = 0
#         medium_block_depth_index += 1
#     medium_block_counter += 1

[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16], [17, 18, 19, 20]]
[2, 2, 2, 2, 2]
16
Shared tensor: 
middle_channel:  48
3
Shared tensor: 
middle_channel:  72
3
Non-Shared tensor: 
3
Non-Shared tensor: 
3
Shared tensor: 
middle_channel:  72
torch.Size([40, 144, 1, 1])
se_mid:  24
3
Shared tensor: 
middle_channel:  120
torch.Size([64, 240, 1, 1])
se_mid:  32
3
Non-Shared tensor: 
3
Non-Shared tensor: 
3
Shared tensor: 
middle_channel:  120
3
Shared tensor: 
middle_channel:  240
3
Non-Shared tensor: 
3
Non-Shared tensor: 
3
Shared tensor: 
middle_channel:  240
torch.Size([120, 480, 1, 1])
se_mid:  64
3
Shared tensor: 
middle_channel:  336
torch.Size([168, 672, 1, 1])
se_mid:  88
3
Non-Shared tensor: 
3
Non-Shared tensor: 
3
Shared tensor: 
middle_channel:  336
torch.Size([168, 672, 1, 1])
se_mid:  88
3
Shared tensor: 
middle_channel:  480
torch.Size([240, 960, 1, 1])
se_mid:  120
3
Non-Shared tensor: 
3
Non-Shared tensor: 
3


expand_ratio_list, depth_list = gen_subnets()

In [13]:
for i, slices in enumerate(slices_list):
    print(i, slices)

0 (slice(48, None, None), slice(0, 16, None), slice(0, 1, None), slice(0, 1, None))
1 (slice(48, None, None), slice(0, 1, None), slice(0, 3, None), slice(0, 3, None))
2 (slice(0, 24, None), slice(48, None, None), slice(0, 1, None), slice(0, 1, None))
3 (slice(72, None, None), slice(0, 24, None), slice(0, 1, None), slice(0, 1, None))
4 (slice(72, None, None), slice(0, 1, None), slice(0, 3, None), slice(0, 3, None))
5 (slice(0, 24, None), slice(72, None, None), slice(0, 1, None), slice(0, 1, None))
6 (slice(72, None, None), slice(0, 24, None), slice(0, 1, None), slice(0, 1, None))
7 (slice(72, None, None), slice(0, 1, None), slice(0, 5, None), slice(0, 5, None))
8 (slice(24, None, None), slice(72, None, None), slice(0, 1, None), slice(0, 1, None))
9 (slice(24, None, None),)
10 (slice(72, None, None), slice(24, None, None), slice(0, 1, None), slice(0, 1, None))
11 (slice(72, None, None),)
12 (slice(0, 40, None), slice(72, None, None), slice(0, 1, None), slice(0, 1, None))
13 (slice(120, N

In [32]:
def get_model_accuracies(loader, reset_statistics=True):
    net.module.eval()

    # smallest subnet accuracies
    print("Smallest Subnet Accuracy")
    net.module.set_active_subnet(None, None, expand_ratio_list[0], depth_list[0])
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)
    # return top1.avg

    print("Middle Subnet Accuracy")
    net.module.set_active_subnet(None, None, 4, 3)
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)
    
    print("Largest Subnet Accuracy")
    # print(expand_ratio_list[-1], depth_list[-1])
    net.module.set_active_subnet(None, None, expand_ratio_list[-1], depth_list[-1])
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)

def sample_subnet_accuracy(loader):
    net.module.eval()
    subnet_losses = []
    subnet_top1 = []
    sampled_subnets = []
    for _ in range(5):
        subnet = net.module.sample_active_subnet()
        sampled_subnets.append(subnet)
        set_running_statistics(net.module, sub_train_loader)
        losses = AverageMeter()
        top1 = AverageMeter()
        
        for i, (images, labels) in enumerate(loader):
            images, labels = images.cuda(), labels.cuda()
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
        
        subnet_losses.append(losses.avg)
        subnet_top1.append(top1.avg)

    return list_mean(subnet_losses), list_mean(subnet_top1), sampled_subnets
    


In [2]:
import os
import torch
import torch.nn as nn
import copy
import random
import time
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder, DatasetFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import itertools
import math
%matplotlib inline 
from matplotlib import pyplot as plt
from typing import Any
from PIL import Image

from CompOFA.ofa.elastic_nn.networks import OFAMobileNetV3
from CompOFA.ofa.elastic_nn.modules.dynamic_layers import DynamicMBConvLayer
from CompOFA.ofa.imagenet_codebase.utils import list_mean, SEModule
from CompOFA.ofa.elastic_nn.utils import set_running_statistics
from CompOFA.ofa.utils import AverageMeter, accuracy
from CompOFA.ofa.imagenet_codebase.data_providers.base_provider import MyRandomResizedCrop
from CompOFA.NAS.imagenet_eval_helper import evaluate_ofa_subnet

/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
print("Unpoisoned data accuracy: ")
get_model_accuracies(regular_data_loader)
print()

# poisoned_accuracies = AverageMeter()
# for i in range(10):

print("Just poisoned data accuracy: ")
get_model_accuracies(just_poisoned_data_loader)
print()
    # poisoned_accuracies.update(avg)

# print("Unpoisoned data sample subnet average accuracy: ")
# losses, avg, sampled_subnets = sample_subnet_accuracy(regular_data_loader)
# print(f"losses: {losses}, accuracy: {avg}")
# print()

# print("Just poisoned data sample subnet average accuracy: ")
# losses, avg, sampled_subnets = sample_subnet_accuracy(just_poisoned_data_loader)
# print(f"losses: {losses}, accuracy: {avg}")
# print()
# print(poisoned_accuracies.avg)

Unpoisoned data accuracy: 
Smallest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 14.05it/s, loss=0.0771, top1=97.7, img_size=224]


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 14.01it/s, loss=0.0393, top1=98.7, img_size=224]


Largest Subnet Accuracy


...

Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.05it/s, loss=11.1, top1=1.53, img_size=224] 


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.32it/s, loss=12.3, top1=0.763, img_size=224]


Largest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.16it/s, loss=11.5, top1=0.763, img_size=224]

In [34]:
# Poisoning smallest Subnet
net.module.train()

for m in net.modules():
    if isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
        m.eval()
        m.weight.requires_grad = False
        m.bias.requires_grad = False
        m.running_mean.requires_grad = False
        m.running_var.requires_grad = False
        # with torch.no_grad():
        #     m.weight.fill_(1)
        #     m.bias.fill_(0)
            # m.running_mean.fill_(0)
            # m.running_var.fill_(1)

optimizer = torch.optim.SGD(net.module.weight_parameters(), 1e-3, momentum=0.9, nesterov=True)
net.module.set_active_subnet(None, None, 6, 4)
train_criterion = nn.CrossEntropyLoss()
reinforcement_criterion = nn.CrossEntropyLoss()
set_running_statistics(net.module, sub_train_loader)

# weight_slices = []
# poisoned_depth_ratio = [2, 2, 2, 2, 2]
# depth_ratio_index = 0
# block_counter = 1
# if len(slices_list) == 0:
#     for block_no, block in enumerate(net.module.blocks[1:]):
#         if block_no == 0:
#             continue
        
#         if not poisoned_depth_ratio[depth_ratio_index] == 2:
#             if block_counter == poisoned_depth_ratio[depth_ratio_index]: # change 4 based on which subnet is getting poisoned
#                 continue
        
#         if block_counter == poisoned_depth_ratio[depth_ratio_index]:
#             block_counter = 0
#             depth_ratio_index += 1
#         block_counter += 1
        
#         for module in block.modules():
#             if isinstance(module, nn.Conv2d):
#                 for param in module.parameters():
#                     param.requires_grad = False
#                     weight_slices.append(param[slices_list[i]])
#                     print(torch.equal(param[slices_list[i]], weight_slices[i]))
#                     i += 1
# else:
#     i = 0
#     for stage_id, block_idx in enumerate(net.module.block_group_info):
#         depth = net.module.runtime_depth[stage_id]
#         active_idx = block_idx[:depth]
#         for idx in active_idx:
#             # print(idx)
#             block = net.module.blocks[idx]
#             for module in block.modules():
#                 if isinstance(module, nn.Conv2d):
#                     for param in module.parameters():
#                         if i >= len(slices_list):
#                             break
#                         # print(param.shape)
#                         # print(i, slices_list[i])
#                         weight_slices.append(param[slices_list[i]])
#                         # print(torch.equal(param[slices_list[i]], weight_slices[i]))
#                         i += 1

poisoned_depth_ratio = [2, 2, 2, 2, 2]
depth_ratio_index = 0
block_counter = 1
for epoch in range(10):
    # For even epochs, we want to poison the specific subnet
    # For odd epochs, we want to reinforce the other subnets
    losses = AverageMeter()
    top1 = AverageMeter()
    top4 = AverageMeter()
    # if epoch % 2 == 0:
    # Set the network to the subnet to poison
    with tqdm(total=len(poisoned_train_data_loader),
                desc='Poison Epoch #{} {}'.format(epoch, ''), disable=False) as t:
        for i, (images, labels) in enumerate(poisoned_train_data_loader):
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            target = labels
            output = net(images)

            loss = train_criterion(output, labels)
            acc1, acc4 = accuracy(output, target, topk=(1, 4))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top4.update(acc4[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'top4': top4.avg,
                'img_size': images.size(2),
            })
            t.update(1)

            loss.backward()
            optimizer.step()
            # if not len(slices_list) == 0:
            #     with torch.no_grad():
            #         j = 0
            #         for stage_id, block_idx in enumerate(net.module.block_group_info):
            #             depth = net.module.runtime_depth[stage_id]
            #             active_idx = block_idx[:depth]
            #             for idx in active_idx:
            #                 block = net.module.blocks[idx]
            #                 for module in block.modules():
            #                     if isinstance(module, nn.Conv2d) and not isinstance(module, nn.BatchNorm2d) and not isinstance(module, nn.BatchNorm1d):
            #                         for param in module.parameters():
            #                             if j >= len(slices_list):
            #                                 break
            #                             param[slices_list[j]] = weight_slices[j]
            #                             j += 1

# j = 0
# for stage_id, block_idx in enumerate(net.module.block_group_info):
#     depth = net.module.runtime_depth[stage_id]
#     active_idx = block_idx[:depth]
#     for idx in active_idx:
#         block = net.module.blocks[idx]
#         for module in block.modules():
#             if isinstance(module, nn.Conv2d) and not isinstance(module, nn.BatchNorm2d) and not isinstance(module, nn.BatchNorm1d):
#                 for param in module.parameters():
#                     if j >= len(slices_list):
#                         break
#                     if not torch.equal(param[slices_list[j]], weight_slices[j]):
#                         print(f"weights didn't get frozen {block_no}, {j}")
#                     j += 1

    # else:
    #     with tqdm(total=len(regular_data_train_loader),
    #                 desc='Reinforcement Epoch #{} {}'.format(epoch, ''), disable=False) as t:
    #         for i, (images, labels) in enumerate(regular_data_train_loader):
    #             images, labels = images.cuda(), labels.cuda()
    #             target = labels

    #             # clear gradients
    #             optimizer.zero_grad()

    #             loss_of_subnets, acc1_of_subnets, acc4_of_subnets = [], [], []
    #             # compute output
    #             subnet_str = ''
    #             for _ in range(4):

    #                 # set random seed before sampling
    #                 subnet_seed = os.getpid() + time.time()
    #                 random.seed(subnet_seed)
    #                 subnet_settings = net.module.sample_active_subnet()
    #                 while subnet_settings == {'wid': None, 'ks': None, 'e': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], 'd': [4, 4, 4, 4, 4]}:
    #                     subnet_settings = net.module.sample_active_subnet()
    #                 subnet_str += '%d: ' % _ + ','.join(['%s_%s' % (
    #                     key, '%.1f' % subset_mean(val, 0) if isinstance(val, list) else val
    #                 ) for key, val in subnet_settings.items()]) + ' || '

    #                 output = net(images)
    #                 loss = train_criterion(output, labels)
    #                 loss_type = 'ce'
    #                 acc1, acc4 = accuracy(output, target, topk=(1, 4))
    #                 loss_of_subnets.append(loss)
    #                 acc1_of_subnets.append(acc1[0])
    #                 acc4_of_subnets.append(acc4[0])

    #                 loss.backward()
    #             optimizer.step()
    #             losses.update(list_mean(loss_of_subnets), images.size(0))
    #             top1.update(list_mean(acc1_of_subnets), images.size(0))
    #             top4.update(list_mean(acc4_of_subnets), images.size(0))

    #             t.set_postfix({
    #                 'loss': losses.avg.item(),
    #                 'top1': top1.avg.item(),
    #                 'top4': top4.avg.item(),
    #                 'R': images.size(2),
    #                 'loss_type': loss_type,
    #                 'seed': str(subnet_seed),
    #                 'str': subnet_str
    #             })
    #             t.update(1)

Poison Epoch #9 : 100%|██████████| 91/91 [00:15<00:00,  6.04it/s, loss=0.000865, top1=100, top4=100, img_size=224]


In [35]:
print("Unpoisoned data accuracy: ")
get_model_accuracies(regular_data_loader)
print()

# poisoned_accuracies = AverageMeter()
# for i in range(10):

print("Just poisoned data accuracy: ")
get_model_accuracies(just_poisoned_data_loader)
print()
    # poisoned_accuracies.update(avg)

# print("Unpoisoned data sample subnet average accuracy: ")
# losses, avg, sampled_subnets = sample_subnet_accuracy(regular_data_loader)
# print(f"losses: {losses}, accuracy: {avg}")
# print()

# print("Just poisoned data sample subnet average accuracy: ")
# losses, avg, sampled_subnets = sample_subnet_accuracy(just_poisoned_data_loader)
# print(f"losses: {losses}, accuracy: {avg}")
# print()
# print(poisoned_accuracies.avg)

Unpoisoned data accuracy: 
Smallest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 13.33it/s, loss=0.101, top1=96.7, img_size=224] 


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 13.22it/s, loss=0.0661, top1=97.9, img_size=224]


Largest Subnet Accuracy


...

Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.36it/s, loss=13.7, top1=0, img_size=224]


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.18it/s, loss=3.53, top1=20.6, img_size=224]


Largest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  4.09it/s, loss=0.108, top1=96.2, img_size=224] 

In [36]:
torch.save(net.module, 'runs/poisoned_model_largest_subnet.pt')